# House Stage 1 regression: two-model ensemble, rolling 2018–2022

Two complementary models, blended. The questions: how correlated are their errors, and does a blend whose weight is chosen **without peeking** beat the polling model alone?

- **Model A, polling-informed:** `partisan_lean` + `gen_margin_raw` + `inc_signed` (**fixed at +2.5**) + `partisan_lean:inc_signed` + `gb_momentum_30d_raw` (`run_ols_fixed`).
- **Model B, fundamentals:** `partisan_lean` + `fundraising_advantage` + `rel_state_college` + `pres_party_signed:log_gas_yoy` + `pres_party_signed:is_midterm` (OLS). B uses **no generic-ballot data** (asserted below) and, as specified, **no incumbency term**; `fundraising_advantage` carries much of it.
- **Optional B additions,** each tested: `hispanic`, `pres_party_signed:cpi_region_log_yoy`, and both; and `rel_state_hispanic` (the district's Hispanic share minus its state's, in points), alone and with the CPI term.
- **Model B without `partisan_lean`** (`B_no_lean*`): fundraising, `rel_state_college`, gas and the midterm penalty, plus the same optional additions, and with `rel_state_college` replaced by `rel_college` (district − national college share, points) or `log_rel_college` (log of district ÷ national college share, derived in the models cell from `college` and `rel_college`). It is more different from A by construction; the question is whether that buys a better blend. The ex-ante filter still uses `partisan_lean`, but only to choose which districts are scored.

**The blend** predicts (1 − w) · A + w · B on the districts both predict (`experiments/house/ensemble.py`). It is scored exactly like a single model: the same valid districts and the same `competitive_ex_ante` (|lean + generic ballot| ≤ 10, evaluation only) and `competitive_ex_post` (|margin| ≤ 7.5) subsets. A and B are fit on the same rows in every window.

**No-peeking weights:** the weight for eval year *t* is the one with the best mean `WEIGHT_METRIC` on `WEIGHT_SUBSET` over the **earlier windows only** (`ensemble.choose_weight`; ties go to the smaller w):

| eval year | models trained on | weight chosen on |
|---|---|---|
| 2018 | 2006–2016 | 2016 (models trained on 2006–2014) |
| 2020 | 2006–2018 | 2016, 2018 |
| 2022 | 2006–2020 | 2016, 2018, 2020 |

The 2016 window exists only to pick 2018's weight and is never reported as a result. A fixed grid of weights is also shown, as in-sample context, since it looks at the scored years.

**Caveats:**

- **FEC timing:** `fundraising_advantage`'s historical totals run through December 31, after the election, so B's skill, and the blend's, are somewhat optimistic for a truly pre-election model.
- **Few years:** with three eval years and one or three validation years per weight, the chosen weights are noisy.
- **2024** is never used (`USE_HOLDOUT = False`).

In [1]:
%load_ext autoreload
%autoreload 2
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown

# The repository root is the first folder up from here that holds pyproject.toml, so this works at any depth
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT))  # repository root, for experiments.house
import house.regression as hr  # hr.<Tab> lists everything; names are in src/house/regression/__init__.py
from experiments.house import ensemble, evaluation, tracking
from experiments.house.run import run_ols_fixed

In [2]:
import warnings

with warnings.catch_warnings(action="ignore"):  # loading warns with the same summaries printed below
    predictors = hr.load_predictors()  # data/processed/house/modeling.csv, every year (2006-2026)
    targets = hr.load_targets()        # data/processed/house/targets.csv, every year (2006-2024)
print(hr.validate_schema(predictors).summary())  # .issues lists every affected key

In [3]:
TARGET = "adj_margin_2p_norm"
ROLLING_WINDOWS = {  # eval year: its expanding training window
    2018: [2006, 2008, 2010, 2012, 2014, 2016],
    2020: [2006, 2008, 2010, 2012, 2014, 2016, 2018],
    2022: [2006, 2008, 2010, 2012, 2014, 2016, 2018, 2020],
}
# plus one earlier window, only to choose the blend weight for 2018 without peeking: it is never scored as a result
VALIDATION_WINDOWS = {2016: [2006, 2008, 2010, 2012, 2014]}
WINDOWS = {**VALIDATION_WINDOWS, **ROLLING_WINDOWS}
USE_HOLDOUT = False     # never True here: no window touches HOLDOUT_YEARS (2024)
# Evaluation only (training is unaffected): observed, non-imputed targets of these race types are scored ("valid")
EVAL_RACE_TYPES = ["dr", "dr_multi", "dr_jungle", "dr_rcv", "dr_writein"]
# Competitive subsets of the valid targets: name -> (column, threshold); a row is in one when abs(column) <= threshold.
# column is a column name or arithmetic of them, from the targets or the predictors
EX_ANTE_THRESHOLD = 10   # points: any non-negative int or float, e.g. 5, 7, or 7.5
EX_POST_THRESHOLD = 7.5
COMPETITIVE = {
    # expected margin under uniform swing: two-party lean (2 x cook_pvi) + eve-of-election two-party generic ballot
    "competitive_ex_ante": ("partisan_lean + gb_margin_2p_norm", EX_ANTE_THRESHOLD),
    "competitive_ex_post": ("margin_2p_norm", EX_POST_THRESHOLD),  # observed two-party margin
}
SUMMARY_SECTIONS = {"ALL": "valid", "EX_ANTE_COMPETITIVE": "competitive_ex_ante",
                    "EX_POST_COMPETITIVE": "competitive_ex_post"}
RANK_BY = "mae"     # variants ranked by mean out-of-sample MAE ...
TIEBREAK = "rmse"   # ... ties broken by RMSE

## Models

In [4]:
FIXED_TERM = "inc_signed"
MODEL_A_FIXED_INC = 2.5  # imposed incumbency coefficient in Model A, points of D margin

# Known before every election: the White House party (+1 D, -1 R) and whether the election is a midterm
PRES_PARTY_SIGNED = {2006: -1, 2008: -1, 2010: 1, 2012: 1, 2014: 1, 2016: 1, 2018: -1, 2020: -1, 2022: 1, 2024: 1,
                     2026: -1}
predictors = predictors.assign(pres_party_signed=predictors["year"].map(PRES_PARTY_SIGNED),
                               is_midterm=(predictors["year"] % 4 == 2).astype(int))
# log(district / national college share), in the form of log_rel_income. The national share isn't stored, but
# rel_college = college - national, so college - rel_college is each year's national share (the same for every
# district of a year; the median guards against rounding)
national_college = (predictors["college"] - predictors["rel_college"]).groupby(predictors["year"]).transform("median")
predictors = predictors.assign(log_rel_college=np.log(predictors["college"] / national_college))

MODEL_A_NAME = "A_polling"
MODEL_A = (f"{TARGET} ~ partisan_lean + gen_margin_raw + inc_signed + partisan_lean:inc_signed"
           " + gb_momentum_30d_raw")
MODEL_B = (f"{TARGET} ~ partisan_lean + fundraising_advantage + rel_state_college"
           " + pres_party_signed:log_gas_yoy + pres_party_signed:is_midterm")
B_ADDITIONS = {  # Model B variant: the terms it adds
    "B_fundamentals": [],
    "B_plus_hispanic": ["hispanic"],
    "B_plus_cpi": ["pres_party_signed:cpi_region_log_yoy"],
    "B_plus_hispanic_cpi": ["hispanic", "pres_party_signed:cpi_region_log_yoy"],
    # the Hispanic share relative to the district's state (district - state, points; acs_supplement.py)
    "B_plus_rel_hispanic": ["rel_state_hispanic"],
    "B_plus_rel_hispanic_cpi": ["rel_state_hispanic", "pres_party_signed:cpi_region_log_yoy"],
}
MODEL_B_VARIANTS = {name: MODEL_B + "".join(f" + {term}" for term in terms) for name, terms in B_ADDITIONS.items()}

# Model B with no partisan_lean at all: its district-level signal comes from fundraising and demographics alone
MODEL_B_NO_LEAN = (f"{TARGET} ~ fundraising_advantage + rel_state_college"
                   " + pres_party_signed:log_gas_yoy + pres_party_signed:is_midterm")
NO_LEAN_ADDITIONS = {  # no-lean Model B variant: the terms it adds
    "B_no_lean": [],
    "B_no_lean_plus_hispanic": ["hispanic"],
    "B_no_lean_plus_cpi": ["pres_party_signed:cpi_region_log_yoy"],
    "B_no_lean_plus_hispanic_cpi": ["hispanic", "pres_party_signed:cpi_region_log_yoy"],
}
MODEL_B_VARIANTS |= {name: MODEL_B_NO_LEAN + "".join(f" + {term}" for term in terms)
                     for name, terms in NO_LEAN_ADDITIONS.items()}
NO_LEAN_COLLEGE = {  # no-lean Model B with another college measure in place of rel_state_college
    "B_no_lean_rel_college": "rel_college",          # district - national college share, points
    "B_no_lean_log_rel_college": "log_rel_college",  # log(district / national college share)
}
MODEL_B_VARIANTS |= {name: MODEL_B_NO_LEAN.replace("rel_state_college", column)
                     for name, column in NO_LEAN_COLLEGE.items()}

# Guard: no generic-ballot column in any Model B
GB_COLUMNS = ["gen_margin_raw", *(column for column in predictors.columns if column.startswith("gb_"))]
for formula in MODEL_B_VARIANTS.values():
    _, variables = evaluation.formula_variables(formula, predictors.columns)
    assert not set(variables) & set(GB_COLUMNS), f"generic-ballot column in a Model B: {formula}"

# A and every B share the same rows: every column any of them uses
SHARED_COLUMNS = evaluation.shared_columns(formulas=[MODEL_A, *MODEL_B_VARIANTS.values()], columns=predictors.columns)
print("shared columns:", SHARED_COLUMNS)
for name, formula in {MODEL_A_NAME: MODEL_A, **MODEL_B_VARIANTS}.items():
    print(f"{name:22s} {formula}")

shared columns: ['partisan_lean', 'gen_margin_raw', 'inc_signed', 'gb_momentum_30d_raw', 'fundraising_advantage', 'rel_state_college', 'pres_party_signed', 'log_gas_yoy', 'is_midterm', 'hispanic', 'cpi_region_log_yoy', 'rel_state_hispanic', 'rel_college', 'log_rel_college']
A_polling              adj_margin_2p_norm ~ partisan_lean + gen_margin_raw + inc_signed + partisan_lean:inc_signed + gb_momentum_30d_raw
B_fundamentals         adj_margin_2p_norm ~ partisan_lean + fundraising_advantage + rel_state_college + pres_party_signed:log_gas_yoy + pres_party_signed:is_midterm
B_plus_hispanic        adj_margin_2p_norm ~ partisan_lean + fundraising_advantage + rel_state_college + pres_party_signed:log_gas_yoy + pres_party_signed:is_midterm + hispanic
B_plus_cpi             adj_margin_2p_norm ~ partisan_lean + fundraising_advantage + rel_state_college + pres_party_signed:log_gas_yoy + pres_party_signed:is_midterm + pres_party_signed:cpi_region_log_yoy
B_plus_hispanic_cpi    adj_margin_2p_norm ~

In [5]:
# Every window (the 2016 validation window included): Model A and each Model B, all on the same rows. The
# missing-predictor warnings (FEC has no data for a few winners) repeat; each run's exclusions are in its .excluded
with warnings.catch_warnings(action="ignore"):
    fits = {}
    for eval_year, train_years in WINDOWS.items():
        model_a = run_ols_fixed(
            formula=MODEL_A,
            train_years=train_years,
            eval_years=[eval_year],
            predict_years=[],
            fixed={FIXED_TERM: MODEL_A_FIXED_INC},
            predictors=predictors,
            targets=targets,
            use_holdout=USE_HOLDOUT,
            require_complete=SHARED_COLUMNS,
            race_types=EVAL_RACE_TYPES,
            competitive=COMPETITIVE,
            name=MODEL_A_NAME,
            save=False,
        )
        models_b = [
            run_ols_fixed(
                formula=formula,
                train_years=train_years,
                eval_years=[eval_year],
                predict_years=[],
                fixed={},  # OLS
                predictors=predictors,
                targets=targets,
                use_holdout=USE_HOLDOUT,
                require_complete=SHARED_COLUMNS,
                race_types=EVAL_RACE_TYPES,
                competitive=COMPETITIVE,
                name=name,
                save=False,
            )
            for name, formula in MODEL_B_VARIANTS.items()
        ]
        fits[eval_year] = {"A_vs_B": [model_a, *models_b]}  # Model A first: the base of every delta

for eval_year, blocks in fits.items():  # matched samples
    rows = {(run.counts["train_rows_hash"], run.counts["eval_rows_hash"]) for run in blocks["A_vs_B"]}
    assert len(rows) == 1, f"{eval_year}: models use different rows"
rolling = {year: fits[year] for year in ROLLING_WINDOWS}  # the scored windows; 2016 only chooses 2018's weight
model = {year: {run.config["name"]: run for run in blocks["A_vs_B"]} for year, blocks in fits.items()}

## Each model alone

Model A and each Model B on its own, by eval year and mean over 2018, 2020 and 2022. Δ is B − A on the same districts. B is expected to be worse: what matters for a blend is whether its errors differ from A's.

In [6]:
for section, subset in SUMMARY_SECTIONS.items():
    by_year = pd.concat(
        {year: tracking.best_of_blocks(blocks, subset=subset, rank_by=RANK_BY, top=None, tiebreak=TIEBREAK)[0]
         for year, blocks in rolling.items()}, names=["eval_year"]).droplevel("block")
    mean, _ = tracking.rolling_best_of(rolling, subset=subset, rank_by=RANK_BY, top=None, tiebreak=TIEBREAK)
    display(Markdown(f"### {section} (`{subset}`): by eval year"))
    display(tracking.style_best_of(by_year))
    display(Markdown(f"### {section} (`{subset}`): mean of {', '.join(map(str, ROLLING_WINDOWS))}"))
    display(tracking.style_best_of(mean.droplevel("block")))

### ALL (`valid`): by eval year

### ALL (`valid`): mean of 2018, 2020, 2022

,n_train,n,mae,rmse,bias,|bias|,accuracy_pct,precision_D_pct,precision_R_pct,missed_races,missed_pct,mae_2018,mae_2020,mae_2022,Δmae,Δrmse,Δaccuracy_pct,Δprecision_D_pct,Δprecision_R_pct,Δmissed_races,Δmissed_pct,Δ|bias|
run,,,,,,,,,,,,,,,,,,,,,,
A_polling,2680,400,5.59,7.21,-0.25,1.49,95.66,95.77,95.75,17,4.34,6.10,5.54,5.14,-,-,-,-,-,-,-,-
B_fundamentals,2680,400,6.59,8.16,-1.38,3.27,93.64,92.33,95.46,25,6.36,7.84,5.57,6.37,+1.00 ✗,+0.95 ✗,-2.02 ✗,-3.45 ✗,-0.28 ✗,+8 ✗,+2.02 ✗,+1.78 ✗
B_plus_rel_hispanic,2680,400,6.59,8.17,-1.38,3.27,93.64,92.33,95.46,25,6.36,7.83,5.58,6.37,+1.00 ✗,+0.96 ✗,-2.02 ✗,-3.45 ✗,-0.28 ✗,+8 ✗,+2.02 ✗,+1.78 ✗
B_plus_hispanic,2680,400,6.61,8.19,-1.40,3.30,93.64,92.33,95.46,25,6.36,7.87,5.58,6.37,+1.02 ✗,+0.98 ✗,-2.02 ✗,-3.45 ✗,-0.28 ✗,+8 ✗,+2.02 ✗,+1.81 ✗
B_plus_cpi,2680,400,6.94,8.56,-1.24,4.50,93.48,92.68,94.97,26,6.52,7.55,6.05,7.20,+1.34 ✗,+1.35 ✗,-2.18 ✗,-3.10 ✗,-0.77 ✗,+9 ✗,+2.18 ✗,+3.01 ✗
B_plus_rel_hispanic_cpi,2680,400,6.94,8.57,-1.24,4.49,93.48,92.68,94.97,26,6.52,7.55,6.05,7.20,+1.34 ✗,+1.35 ✗,-2.18 ✗,-3.10 ✗,-0.77 ✗,+9 ✗,+2.18 ✗,+3.00 ✗
B_plus_hispanic_cpi,2680,400,6.95,8.60,-1.27,4.53,93.56,92.69,95.14,26,6.44,7.58,6.08,7.20,+1.36 ✗,+1.38 ✗,-2.10 ✗,-3.08 ✗,-0.60 ✗,+8 ✗,+2.10 ✗,+3.04 ✗
B_no_lean_rel_college,2680,400,12.31,15.46,-3.86,4.52,86.58,80.79,96.89,54,13.42,13.53,12.82,10.57,+6.71 ✗,+8.25 ✗,-9.08 ✗,-14.98 ✗,+1.14 ✓,+36 ✗,+9.08 ✗,+3.03 ✗
B_no_lean_plus_hispanic,2680,400,12.34,15.42,-4.19,4.58,86.99,80.72,97.59,52,13.01,13.53,12.89,10.61,+6.75 ✗,+8.20 ✗,-8.67 ✗,-15.05 ✗,+1.85 ✓,+35 ✗,+8.67 ✗,+3.09 ✗


### EX_ANTE_COMPETITIVE (`competitive_ex_ante`): by eval year

### EX_ANTE_COMPETITIVE (`competitive_ex_ante`): mean of 2018, 2020, 2022

,n_train,n,mae,rmse,bias,|bias|,accuracy_pct,precision_D_pct,precision_R_pct,missed_races,missed_pct,mae_2018,mae_2020,mae_2022,Δmae,Δrmse,Δaccuracy_pct,Δprecision_D_pct,Δprecision_R_pct,Δmissed_races,Δmissed_pct,Δ|bias|
run,,,,,,,,,,,,,,,,,,,,,,
A_polling,2680,91,4.96,6.30,-0.34,1.77,84.77,85.77,84.72,14,15.23,4.77,4.81,5.31,-,-,-,-,-,-,-,-
B_plus_rel_hispanic,2680,91,5.19,6.43,-1.28,2.47,77.22,74.50,81.31,21,22.78,6.12,4.27,5.19,+0.23 ✗,+0.12 ✗,-7.56 ✗,-11.27 ✗,-3.41 ✗,+7 ✗,+7.56 ✗,+0.71 ✗
B_plus_hispanic,2680,91,5.19,6.42,-1.30,2.51,77.22,74.50,81.31,21,22.78,6.14,4.27,5.17,+0.23 ✗,+0.12 ✗,-7.56 ✗,-11.27 ✗,-3.41 ✗,+7 ✗,+7.56 ✗,+0.74 ✗
B_fundamentals,2680,91,5.20,6.43,-1.30,2.49,77.22,74.50,81.31,21,22.78,6.14,4.27,5.19,+0.24 ✗,+0.12 ✗,-7.56 ✗,-11.27 ✗,-3.41 ✗,+7 ✗,+7.56 ✗,+0.72 ✗
B_plus_rel_hispanic_cpi,2680,91,5.45,6.78,-1.16,3.67,76.38,74.92,80.77,21,23.62,5.79,4.68,5.87,+0.48 ✗,+0.48 ✗,-8.39 ✗,-10.85 ✗,-3.95 ✗,+8 ✗,+8.39 ✗,+1.90 ✗
B_plus_hispanic_cpi,2680,91,5.45,6.79,-1.19,3.70,76.75,75.11,81.39,21,23.25,5.79,4.70,5.87,+0.49 ✗,+0.48 ✗,-8.03 ✗,-10.66 ✗,-3.33 ✗,+7 ✗,+8.03 ✗,+1.94 ✗
B_plus_cpi,2680,91,5.46,6.79,-1.18,3.68,76.38,74.92,80.77,21,23.62,5.81,4.69,5.87,+0.50 ✗,+0.48 ✗,-8.39 ✗,-10.85 ✗,-3.95 ✗,+8 ✗,+8.39 ✗,+1.92 ✗
B_no_lean,2680,91,9.46,11.03,-6.96,6.96,66.20,58.89,90.22,31,33.80,10.64,10.72,7.03,+4.50 ✗,+4.73 ✗,-18.58 ✗,-26.88 ✗,+5.50 ✓,+18 ✗,+18.58 ✗,+5.19 ✗
B_no_lean_log_rel_college,2680,91,9.55,11.14,-7.08,7.08,66.61,59.35,90.55,31,33.39,10.74,10.79,7.14,+4.59 ✗,+4.84 ✗,-18.17 ✗,-26.42 ✗,+5.83 ✓,+17 ✗,+18.17 ✗,+5.32 ✗


### EX_POST_COMPETITIVE (`competitive_ex_post`): by eval year

### EX_POST_COMPETITIVE (`competitive_ex_post`): mean of 2018, 2020, 2022

,n_train,n,mae,rmse,bias,|bias|,accuracy_pct,precision_D_pct,precision_R_pct,missed_races,missed_pct,mae_2018,mae_2020,mae_2022,Δmae,Δrmse,Δaccuracy_pct,Δprecision_D_pct,Δprecision_R_pct,Δmissed_races,Δmissed_pct,Δ|bias|
run,,,,,,,,,,,,,,,,,,,,,,
A_polling,2680,60,6.26,8.21,+0.22,2.39,73.23,75.02,74.89,16,26.77,7.47,5.74,5.58,-,-,-,-,-,-,-,-
B_fundamentals,2680,60,5.20,6.53,-1.09,2.50,65.84,64.52,70.97,21,34.16,6.23,4.20,5.18,-1.06 ✓,-1.68 ✓,-7.39 ✗,-10.49 ✗,-3.92 ✗,+5 ✗,+7.39 ✗,+0.11 ✗
B_plus_rel_hispanic,2680,60,5.22,6.59,-1.10,2.51,65.84,64.52,70.97,21,34.16,6.25,4.24,5.18,-1.04 ✓,-1.62 ✓,-7.39 ✗,-10.49 ✗,-3.92 ✗,+5 ✗,+7.39 ✗,+0.12 ✗
B_plus_hispanic,2680,60,5.24,6.60,-1.13,2.55,65.84,64.52,70.97,21,34.16,6.31,4.24,5.17,-1.02 ✓,-1.61 ✓,-7.39 ✗,-10.49 ✗,-3.92 ✗,+5 ✗,+7.39 ✗,+0.17 ✗
B_plus_cpi,2680,60,5.49,6.95,-0.96,3.68,62.92,63.89,69.58,22,37.08,5.92,4.60,5.93,-0.78 ✓,-1.26 ✓,-10.31 ✗,-11.13 ✗,-5.31 ✗,+6 ✗,+10.31 ✗,+1.29 ✗
B_plus_rel_hispanic_cpi,2680,60,5.50,7.00,-0.97,3.69,62.92,63.89,69.58,22,37.08,5.95,4.63,5.93,-0.76 ✓,-1.21 ✓,-10.31 ✗,-11.13 ✗,-5.31 ✗,+6 ✗,+10.31 ✗,+1.31 ✗
B_plus_hispanic_cpi,2680,60,5.54,7.04,-1.03,3.75,63.47,64.19,70.93,22,36.53,6.00,4.68,5.93,-0.73 ✓,-1.17 ✓,-9.76 ✗,-10.83 ✗,-3.96 ✗,+6 ✗,+9.76 ✗,+1.36 ✗
B_no_lean,2680,60,8.96,10.61,-7.31,7.31,54.36,53.61,77.12,28,45.64,10.53,10.00,6.33,+2.69 ✗,+2.40 ✗,-18.87 ✗,-21.41 ✗,+2.24 ✓,+12 ✗,+18.87 ✗,+4.92 ✗
B_no_lean_log_rel_college,2680,60,9.03,10.68,-7.38,7.38,55.50,54.53,82.22,27,44.50,10.64,10.01,6.43,+2.76 ✗,+2.47 ✗,-17.74 ✗,-20.49 ✗,+7.33 ✓,+11 ✗,+17.74 ✗,+4.99 ✗


## How correlated are the errors?

The correlation of A's and B's residuals (actual − predicted) on the same districts, by eval year and pooled over the three years, for each subset. Each model's bias and MAE are shown alongside.

- **Within a year,** the correlation is about district-level errors. Lower means more to gain from blending.
- **Pooled over years,** it also carries each year's national miss, the `bias` columns. Year-level errors that differ in sign are what a blend can average away.

In [7]:
SUBSETS = list(SUMMARY_SECTIONS.values())

pairs = {}  # (Model B, eval year or "pooled"): the (A, B) results it is computed on
for b_name in MODEL_B_VARIANTS:
    for year in ROLLING_WINDOWS:
        pairs[(b_name, year)] = [(model[year][MODEL_A_NAME], model[year][b_name])]
    pairs[(b_name, "pooled")] = [(model[year][MODEL_A_NAME], model[year][b_name]) for year in ROLLING_WINDOWS]
correlations = pd.DataFrame({key: ensemble.error_correlation(results, SUBSETS).stack() for key, results in pairs.items()}).T
correlations.index.names = ["model_b", "eval_year"]
correlations.style.format({column: "{:.0f}" if column[1] == "n" else "{:+.2f}" if column[1].startswith("bias")
                           else "{:.2f}" for column in correlations.columns})

## Blend weights: the fixed grid (in-sample context)

The blend's metrics at each fixed weight w on B (0 is Model A alone, 1 is B alone), as a mean over the three eval years. The grid looks at the scored years, so its best weight is **not** a fair estimate; the no-peeking blend follows.

In [8]:
WEIGHT_GRID = np.round(np.arange(0, 1.0001, 0.05), 2)  # w on Model B

with warnings.catch_warnings(action="ignore"):
    scores = {  # Model B variant: {window: the blend's metrics by (weight, subset)}, every window including 2016
        b_name: {year: ensemble.blend_scores(model[year][MODEL_A_NAME], model[year][b_name], WEIGHT_GRID, targets,
                                             predictors, TARGET, COMPETITIVE, EVAL_RACE_TYPES)
                 for year in WINDOWS}
        for b_name in MODEL_B_VARIANTS
    }

GRID_METRICS = ["mae", "rmse", "bias", "accuracy_pct"]
for b_name in MODEL_B_VARIANTS:
    mean = sum(scores[b_name][year] for year in ROLLING_WINDOWS) / len(ROLLING_WINDOWS)
    grid = pd.concat({section: mean.xs(subset, level="subset")[GRID_METRICS]
                      for section, subset in SUMMARY_SECTIONS.items()}, axis=1)
    display(Markdown(f"### A + w · {b_name}: mean of {', '.join(map(str, ROLLING_WINDOWS))} (in-sample)"))
    display(grid.style.format(precision=2).highlight_min(
        subset=[c for c in grid.columns if c[1] in ("mae", "rmse")], props="font-weight: bold").highlight_max(
        subset=[c for c in grid.columns if c[1] == "accuracy_pct"], props="font-weight: bold"))

### A + w · B_fundamentals: mean of 2018, 2020, 2022 (in-sample)

### A + w · B_plus_hispanic: mean of 2018, 2020, 2022 (in-sample)

### A + w · B_plus_cpi: mean of 2018, 2020, 2022 (in-sample)

### A + w · B_plus_hispanic_cpi: mean of 2018, 2020, 2022 (in-sample)

### A + w · B_plus_rel_hispanic: mean of 2018, 2020, 2022 (in-sample)

### A + w · B_plus_rel_hispanic_cpi: mean of 2018, 2020, 2022 (in-sample)

... (2 outputs left out)

### A + w · B_no_lean_plus_cpi: mean of 2018, 2020, 2022 (in-sample)

### A + w · B_no_lean_plus_hispanic_cpi: mean of 2018, 2020, 2022 (in-sample)

### A + w · B_no_lean_rel_college: mean of 2018, 2020, 2022 (in-sample)

### A + w · B_no_lean_log_rel_college: mean of 2018, 2020, 2022 (in-sample)

## The no-peeking blend

For each eval year, the weight comes from the earlier windows (`ensemble.choose_weight`) and the blend is scored on that year. The `mean` row averages the three eval years, each weighted equally. Δ is blend − Model A alone on the same districts: ✓ marks an improvement and ✗ a regression.

In [9]:
WEIGHT_SUBSET = "competitive_ex_ante"  # the weight minimizes this subset's ...
WEIGHT_METRIC = "mae"                   # ... mean MAE over the earlier windows
BLEND_METRICS = ["mae", "rmse", "bias", "accuracy_pct"]

rows = {}
for b_name in MODEL_B_VARIANTS:
    for year in ROLLING_WINDOWS:
        weight, earlier = ensemble.choose_weight(scores[b_name], year, WEIGHT_SUBSET, WEIGHT_METRIC)
        row = {("weight", "w"): weight, ("weight", "chosen on"): ", ".join(map(str, earlier))}
        for section, subset in SUMMARY_SECTIONS.items():
            blended = scores[b_name][year].loc[(weight, subset), BLEND_METRICS]
            alone = scores[b_name][year].loc[(0.0, subset), BLEND_METRICS]  # w = 0: Model A
            row |= {(section, metric): blended[metric] for metric in BLEND_METRICS}
            row |= {(section, f"Δ{metric}"): blended[metric] - alone[metric] for metric in ["mae", "rmse", "accuracy_pct"]}
        rows[(b_name, year)] = row
no_peek = pd.DataFrame.from_dict(rows, orient="index").rename_axis(["model_b", "eval_year"])
means = no_peek.drop(columns=[("weight", "chosen on")]).groupby(level="model_b", sort=False).mean()
means.index = pd.MultiIndex.from_tuples([(b, "mean") for b in means.index], names=no_peek.index.names)
no_peek = pd.concat([no_peek, means]).sort_index(level="model_b", sort_remaining=False)
no_peek = no_peek.reindex([(b, y) for b in MODEL_B_VARIANTS for y in [*ROLLING_WINDOWS, "mean"]])
display(Markdown(f"### No-peeking blend: weight minimizing mean {WEIGHT_METRIC} on `{WEIGHT_SUBSET}` "
                 "over earlier windows"))
tracking.style_best_of(no_peek)

### No-peeking blend: weight minimizing mean mae on `competitive_ex_ante` over earlier windows

## Coefficients

Each model's coefficients averaged over the three eval windows. In Model A, `inc_signed` is the imposed `MODEL_A_FIXED_INC`. `pres_party_signed:is_midterm` is the president's party's midterm penalty, in points of its margin; a negative value is a penalty.

In [10]:
_, mean_coefficients = tracking.rolling_best_of(rolling, rank_by=RANK_BY, top=None, tiebreak=TIEBREAK)
mean_coefficients.droplevel("block").round(3)

,Intercept,partisan_lean,gen_margin_raw,partisan_lean:inc_signed,gb_momentum_30d_raw,inc_signed,fundraising_advantage,rel_state_college,pres_party_signed:log_gas_yoy,pres_party_signed:is_midterm,hispanic,pres_party_signed:cpi_region_log_yoy,rel_state_hispanic,rel_college,log_rel_college
run,,,,,,,,,,,,,,,
A_polling,-0.389,1.067,1.258,-0.104,-1.462,2.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
B_fundamentals,2.367,0.660,NaN,NaN,NaN,NaN,5.157,-0.004,-0.064,-4.251,NaN,NaN,NaN,NaN,NaN
B_plus_rel_hispanic,2.371,0.659,NaN,NaN,NaN,NaN,5.153,0.001,-0.064,-4.251,NaN,NaN,0.012,NaN,NaN
B_plus_hispanic,2.232,0.658,NaN,NaN,NaN,NaN,5.155,-0.001,-0.063,-4.253,0.009,NaN,NaN,NaN,NaN
B_plus_cpi,2.278,0.662,NaN,NaN,NaN,NaN,5.123,-0.005,-0.016,-3.372,NaN,-0.474,NaN,NaN,NaN
B_plus_rel_hispanic_cpi,2.282,0.662,NaN,NaN,NaN,NaN,5.119,-0.000,-0.016,-3.375,NaN,-0.473,0.011,NaN,NaN
B_plus_hispanic_cpi,2.094,0.661,NaN,NaN,NaN,NaN,5.120,0.000,-0.014,-3.352,0.012,-0.485,NaN,NaN,NaN
B_no_lean_rel_college,1.891,NaN,NaN,NaN,NaN,NaN,9.961,NaN,0.026,-1.736,NaN,NaN,NaN,0.192,NaN
B_no_lean_plus_hispanic,-0.248,NaN,NaN,NaN,NaN,NaN,9.856,0.203,0.030,-1.782,0.136,NaN,NaN,NaN,NaN
